# Siamese U-Net - Building Damage Assessment (xBD)
Train on Kaggle T4 GPU | ResNet-34 encoder | Focal + Dice Loss

In [ ]:
import os

BASE_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train' in dirs and 'test' in dirs:
        BASE_DIR = root
        break
    if root.endswith('train/images') or root.endswith('train\\images'):
        BASE_DIR = os.path.dirname(os.path.dirname(root))
        break

if BASE_DIR is None:
    raise FileNotFoundError("Khong tim thay thu muc train/images trong /kaggle/input/")

print(f"Tim thay thu muc goc dataset tai: {BASE_DIR}")

TRAIN_IMG_DIR = os.path.join(BASE_DIR, 'train', 'images')
TRAIN_LBL_DIR = os.path.join(BASE_DIR, 'train', 'labels')
TEST_IMG_DIR  = os.path.join(BASE_DIR, 'test', 'images')
TEST_LBL_DIR  = os.path.join(BASE_DIR, 'test', 'labels')
OUTPUT_DIR    = '/kaggle/working/checkpoints'
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE   = 512
BATCH_SIZE = 16
NUM_WORKERS = 2
EPOCHS     = 50
LR         = 1e-4
VAL_SPLIT  = 0.15
ENCODER    = 'resnet34'
SEED       = 42
print('Config OK')

In [ ]:
!pip install -q segmentation-models-pytorch albumentations shapely

In [ ]:
import json, random, numpy as np
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image, ImageDraw
from shapely import wkt
from shapely.geometry import mapping
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.notebook import tqdm
from sklearn.metrics import f1_score

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda': print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
DAMAGE_LABEL_MAP = {'no-damage':1,'minor-damage':2,'major-damage':3,'destroyed':4,'un-classified':0}

def json_to_mask(label_path, img_size=1024):
    mask = np.zeros((img_size,img_size), dtype=np.uint8)
    with open(label_path,'r') as f: data=json.load(f)
    feats=data.get('features',{})
    xy=feats.get('xy', feats.get('lng_lat',[]))
    for feat in xy:
        props=feat.get('properties',{})
        label=DAMAGE_LABEL_MAP.get(props.get('subtype','un-classified'),0)
        if label==0: continue
        wkt_str=feat.get('wkt','')
        if not wkt_str: continue
        try:
            geom=wkt.loads(wkt_str)
            coords=list(mapping(geom)['coordinates'][0])
            poly=[(float(x),float(y)) for x,y in coords]
            if len(poly)<3: continue
            pm=Image.fromarray(mask); draw=ImageDraw.Draw(pm)
            draw.polygon(poly, fill=int(label)); mask=np.array(pm)
        except: continue
    return mask

def get_train_transforms(sz):
    return A.Compose([
        A.RandomCrop(sz,sz), A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.OneOf([A.ColorJitter(0.2,0.2,0.2,0.05,p=1), A.RandomBrightnessContrast(p=1)], p=0.5),
        A.GaussianBlur(blur_limit=(3,5),p=0.2),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2(),
    ], additional_targets={'image2':'image','mask2':'mask'})

def get_val_transforms(sz):
    return A.Compose([
        A.CenterCrop(sz,sz),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]), ToTensorV2(),
    ], additional_targets={'image2':'image','mask2':'mask'})

class XBDDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir=image_dir; self.label_dir=label_dir; self.transform=transform
        pre=[f for f in os.listdir(image_dir) if f.endswith('_pre_disaster.png')]
        self.samples=[]
        for p in sorted(pre):
            base=p.replace('_pre_disaster.png','')
            post_img=os.path.join(image_dir, base+'_post_disaster.png')
            post_lbl=os.path.join(label_dir, base+'_post_disaster.json')
            pre_lbl =os.path.join(label_dir, base+'_pre_disaster.json')
            if os.path.exists(post_img) and os.path.exists(post_lbl):
                self.samples.append({'pre_img':os.path.join(image_dir,p),'post_img':post_img,
                                     'pre_lbl':pre_lbl,'post_lbl':post_lbl,'base':base})
    def __len__(self): return len(self.samples)
    def _img(self,p): return np.array(Image.open(p).convert('RGB'),dtype=np.uint8)
    def _masks(self,s):
        dmg=json_to_mask(s['post_lbl'])
        if os.path.exists(s['pre_lbl']):
            with open(s['pre_lbl'],'r') as f: data=json.load(f)
            xy=data.get('features',{}).get('xy', data.get('features',{}).get('lng_lat',[]))
            loc=np.zeros((1024,1024),dtype=np.uint8)
            for feat in xy:
                w=feat.get('wkt','')
                if not w: continue
                try:
                    g=wkt.loads(w); c=list(mapping(g)['coordinates'][0])
                    poly=[(float(x),float(y)) for x,y in c]
                    if len(poly)<3: continue
                    pm=Image.fromarray(loc); d=ImageDraw.Draw(pm); d.polygon(poly,fill=1); loc=np.array(pm)
                except: continue
        else: loc=(dmg>0).astype(np.uint8)
        return loc,dmg
    def __getitem__(self,idx):
        s=self.samples[idx]
        pre=self._img(s['pre_img']); post=self._img(s['post_img'])
        loc,dmg=self._masks(s)
        if self.transform:
            aug=self.transform(image=pre,image2=post,mask=loc,mask2=dmg)
            return {'pre_img':aug['image'],'post_img':aug['image2'],
                    'loc_mask':aug['mask'].long(),'dmg_mask':aug['mask2'].long(),'name':s['base']}
        return {'pre_img':torch.from_numpy(pre.transpose(2,0,1)).float()/255.,
                'post_img':torch.from_numpy(post.transpose(2,0,1)).float()/255.,
                'loc_mask':torch.from_numpy(loc).long(),'dmg_mask':torch.from_numpy(dmg).long(),'name':s['base']}

print('Dataset class OK')

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self,i,o):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(i,o,3,padding=1,bias=False),nn.BatchNorm2d(o),nn.ReLU(True),
                               nn.Conv2d(o,o,3,padding=1,bias=False),nn.BatchNorm2d(o),nn.ReLU(True))
    def forward(self,x): return self.net(x)

class DecoderBlock(nn.Module):
    def __init__(self,i,s,o):
        super().__init__()
        self.up=nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False)
        self.conv=DoubleConv(i+s,o)
    def forward(self,x,skip=None):
        x=self.up(x)
        if skip is not None:
            if x.shape[-2:]!=skip.shape[-2:]: x=F.interpolate(x,size=skip.shape[-2:],mode='bilinear',align_corners=False)
            x=torch.cat([x,skip],dim=1)
        return self.conv(x)

class SiameseUNet(nn.Module):
    def __init__(self,encoder_name='resnet34',encoder_weights='imagenet',n_dmg=5):
        super().__init__()
        self.encoder=smp.encoders.get_encoder(encoder_name,in_channels=3,depth=5,weights=encoder_weights)
        ec=self.encoder.out_channels
        self.bottleneck=DoubleConv(ec[-1]*2,512)
        skip_chs=list(reversed([c*2 for c in ec[1:-1]]))
        dec_out=[256,128,64,32]
        self.decoder=nn.ModuleList()
        inc=512
        for s,o in zip(skip_chs,dec_out): self.decoder.append(DecoderBlock(inc,s,o)); inc=o
        self.final_up=nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False)
        self.final_conv=DoubleConv(32,32)
        self.loc_head=nn.Conv2d(32,2,1)
        self.dmg_head=nn.Conv2d(32,n_dmg,1)
    def forward(self,pre,post):
        pf=self.encoder(pre); qf=self.encoder(post)
        x=self.bottleneck(torch.cat([pf[-1],qf[-1]],dim=1))
        skips=[torch.cat([pf[i],qf[i]],dim=1) for i in range(len(pf)-2,0,-1)]
        for blk,sk in zip(self.decoder,skips): x=blk(x,sk)
        x=self.final_conv(self.final_up(x))
        return self.loc_head(x),self.dmg_head(x)

model=SiameseUNet(ENCODER,'imagenet').to(device)
n=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model OK | Params: {n/1e6:.1f}M')

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self,gamma=2.,weight=None):
        super().__init__(); self.gamma=gamma; self.weight=weight
    def forward(self,logits,targets):
        lp=F.log_softmax(logits,1); pt=torch.exp(lp)
        lpt=lp.gather(1,targets.unsqueeze(1)).squeeze(1)
        ptt=pt.gather(1,targets.unsqueeze(1)).squeeze(1)
        fw=(1-ptt)**self.gamma
        alpha=self.weight.to(logits.device)[targets] if self.weight is not None else 1.
        return (-alpha*fw*lpt).mean()

class DiceLoss(nn.Module):
    def __init__(self,smooth=1.):
        super().__init__(); self.smooth=smooth
    def forward(self,logits,targets):
        nc=logits.shape[1]; p=F.softmax(logits,1)
        oh=F.one_hot(targets,nc).permute(0,3,1,2).float()
        inter=(p*oh).sum((0,2,3)); union=(p+oh).sum((0,2,3))
        return 1-((2*inter+self.smooth)/(union+self.smooth)).mean()

class CombinedLoss(nn.Module):
    DMG_W=torch.tensor([0.2,0.5,2.0,2.5,3.0])
    LOC_W=torch.tensor([0.3,1.5])
    def __init__(self,wl=0.4,wd=0.6,wf=0.6,wdi=0.4):
        super().__init__(); self.wl=wl; self.wd=wd; self.wf=wf; self.wdi=wdi
        self.lf=FocalLoss(2.,self.LOC_W); self.ld=DiceLoss()
        self.df=FocalLoss(2.,self.DMG_W); self.dd=DiceLoss()
    def forward(self,ll,dl,lt,dt):
        lloc=self.wf*self.lf(ll,lt)+self.wdi*self.ld(ll,lt)
        ldmg=self.wf*self.df(dl,dt)+self.wdi*self.dd(dl,dt)
        return self.wl*lloc+self.wd*ldmg

criterion=CombinedLoss()
print('Loss OK')

In [ ]:
all_ds=XBDDataset(TRAIN_IMG_DIR,TRAIN_LBL_DIR,transform=None)
n_total=len(all_ds); n_val=int(n_total*VAL_SPLIT); n_train=n_total-n_val
g=torch.Generator().manual_seed(SEED)
idx=torch.randperm(n_total,generator=g).tolist()
train_idx,val_idx=idx[:n_train],idx[n_train:]

train_ds=Subset(XBDDataset(TRAIN_IMG_DIR,TRAIN_LBL_DIR,get_train_transforms(IMG_SIZE)),train_idx)
val_ds  =Subset(XBDDataset(TRAIN_IMG_DIR,TRAIN_LBL_DIR,get_val_transforms(IMG_SIZE)),  val_idx)

train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True, num_workers=NUM_WORKERS,pin_memory=True,drop_last=True,persistent_workers=True)
val_loader  =DataLoader(val_ds,  batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=True)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} samples')
print(f'Steps/epoch: train={len(train_loader)}, val={len(val_loader)}')

In [ ]:
class MetricAccumulator:
    def __init__(self): self.reset()
    def reset(self): self.lp=[];self.lt=[];self.dp=[];self.dt=[]
    def update(self,ll,dl,lt,dt):
        self.lp.append(ll.argmax(1).cpu().numpy().ravel())
        self.lt.append(lt.cpu().numpy().ravel())
        self.dp.append(dl.argmax(1).cpu().numpy().ravel())
        self.dt.append(dt.cpu().numpy().ravel())
    def compute(self):
        lp=np.concatenate(self.lp); lt=np.concatenate(self.lt)
        dp=np.concatenate(self.dp); dt=np.concatenate(self.dt)
        f1l=f1_score(lt,lp,average='binary',zero_division=0)
        bm=lt==1
        if bm.sum()>0:
            f1d=f1_score(dt[bm],dp[bm],labels=[1,2,3,4],average='macro',zero_division=0)
        else: f1d=0.
        return {'f1_loc':f1l,'f1_dmg':f1d,'score':0.3*f1l+0.7*f1d}
print('Metrics OK')

In [ ]:
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS,eta_min=LR*0.01)
scaler=torch.amp.GradScaler('cuda')

def train_epoch(ep):
    model.train(); total=0; n=0
    for b in tqdm(train_loader,desc=f'Train {ep}',leave=False):
        pre=b['pre_img'].to(device,non_blocking=True)
        post=b['post_img'].to(device,non_blocking=True)
        lg=b['loc_mask'].to(device,non_blocking=True)
        dg=b['dmg_mask'].to(device,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            ll,dl=model(pre,post); loss=criterion(ll,dl,lg,dg)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(),1.)
        scaler.step(optimizer); scaler.update()
        total+=loss.item()*len(pre); n+=len(pre)
    return total/n

@torch.no_grad()
def val_epoch(ep):
    model.eval(); total=0; n=0; acc=MetricAccumulator()
    for b in tqdm(val_loader,desc=f'Val   {ep}',leave=False):
        pre=b['pre_img'].to(device,non_blocking=True)
        post=b['post_img'].to(device,non_blocking=True)
        lg=b['loc_mask'].to(device,non_blocking=True)
        dg=b['dmg_mask'].to(device,non_blocking=True)
        with torch.amp.autocast('cuda'):
            ll,dl=model(pre,post); loss=criterion(ll,dl,lg,dg)
        total+=loss.item()*len(pre); n+=len(pre)
        acc.update(ll,dl,lg,dg)
    return total/n, acc.compute()

best_score=0.; history=[]
for ep in range(1,EPOCHS+1):
    tl=train_epoch(ep)
    vl,m=val_epoch(ep)
    scheduler.step()
    sc=m['score']
    history.append({'epoch':ep,'train_loss':tl,'val_loss':vl,**m})
    print(f'Ep {ep:3d}/{EPOCHS} | Train:{tl:.4f} Val:{vl:.4f} | F1_loc:{m["f1_loc"]:.4f} F1_dmg:{m["f1_dmg"]:.4f} | Score:{sc:.4f}')
    ckpt={'epoch':ep,'model_state':model.state_dict(),'optim_state':optimizer.state_dict(),'score':sc,'metrics':m}
    if ep%5==0: torch.save(ckpt,f'{OUTPUT_DIR}/epoch_{ep:03d}.pth')
    if sc>best_score:
        best_score=sc
        torch.save(ckpt,f'{OUTPUT_DIR}/best_model.pth')
        print(f'New best! {best_score:.4f}')

print(f'Done! Best Score: {best_score:.4f}')

In [ ]:
import matplotlib.pyplot as plt, pandas as pd
df=pd.DataFrame(history)
fig,axes=plt.subplots(1,3,figsize=(15,4))
axes[0].plot(df.epoch,df.train_loss,label='Train'); axes[0].plot(df.epoch,df.val_loss,label='Val')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(df.epoch,df.f1_loc,label='F1_loc'); axes[1].plot(df.epoch,df.f1_dmg,label='F1_dmg')
axes[1].set_title('F1 Scores'); axes[1].legend()
axes[2].plot(df.epoch,df.score,'r-',label='xView2 Score')
axes[2].set_title('xView2 Score'); axes[2].legend()
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/training_curve.png',dpi=150)
plt.show()
print(f'Best checkpoint: {OUTPUT_DIR}/best_model.pth')